# CS781 Phase 7 - Prior Case Retrieval (one notebook, four required runs on the four Phase-1 models)

| Run | Phase-1 model | Adaptation technique | Role |
|---|---|---|---|
| 1 | Arch 4 | **LoRMA** (compulsory), contrastive InfoNCE with BM25 hard negatives | also the KD teacher |
| 2 | Arch 3 | **IA³** (learned rescaling of K, V and FFN activations) | |
| 3 | Arch 2 | **Partial / layer-wise fine-tuning** (last 4 blocks + final norm) | |
| 4 | Arch 1 | **Knowledge distillation**: half-depth student, full fine-tuning, loss = InfoNCE + KL to the Run-1 teacher's query-candidate score distribution + embedding alignment | mandatory KD experiment |
| 5 | Arch 4 | LoRMA with **in-batch negatives only** (optional) | negative-sampling ablation |
| 6 | Arch 4 | LoRMA with a **pairwise margin** objective (optional) | ranking-objective ablation |
| 7 | Arch 1 | same half-depth student, full fine-tuning, **no teacher** (optional) | isolates what KD adds |

**Method.** The Phase-1 models are decoder-only LMs, so each case becomes a dense vector as follows:
1. Sentences are scored and the token budget (`P7_MAX_CHUNKS` x 512 tokens) is filled with lead + tail + SumBasic-ranked middle sentences in original order (same selector as Phase 6).
2. The selected text is packed into sentence-aligned chunks of at most 512 tokens (right-padded, so the causal model never sees a pad token and no attention mask is needed).
3. Each chunk is pooled (mean over tokens, or last token), L2-normalised, and the chunks of a case are aggregated (first chunk / mean / MaxSim).
4. Training is chunk-level contrastive learning with **GradCache**, so in-batch negatives and BM25 hard negatives share one large batch although only a few sequences fit on the GPU at a time.
5. Candidates are ranked by cosine similarity inside the split's own candidate pool; the top 7 are submitted.

Every run is evaluated on Dev with all `pool x centring x aggregation` combinations, and the best combination on Dev is used for Test. A BM25 baseline (no neural weights) is computed for reference and for a tuned dense+BM25 hybrid submission. Only Phase-1 weights are used; no external model or API.

**Kaggle inputs to attach (Add Input)**
1. The Phase 7 data (folder `Phase_7_PCR`), or leave Internet on and the notebook downloads it from the Hugging Face repo like in Phase 6 (needs a Read token with access to the gated repo).
2. The four Phase-1 checkpoint datasets, exactly as in Phases 4/5/6.
3. `phase4-allrun-v4.ipynb` (cells 1-7 are reused for the model code).
4. Optional: a committed Phase-4/5/6 output containing `diagnostics.json`.
5. Run 4 (and Run 7 is independent of it) needs the Run-1 teacher file `Run1_Arch4_LoRMA_trainable.pt`: same session as Run 1, or attach the committed output of the session that ran it.
6. GPU on (T4 x2 or P100). Internet only for the Hugging Face download.

**How to run.** Keep `P7_DRY_RUN = True` first (about 15 minutes, tiny data), then set it to `False`. To use two GPU sessions at once: session A `P7_RUNS = [1, 4]`, session B `P7_RUNS = [2, 3]`; outputs go to `/kaggle/working/p7_out` (Save & Commit). Run the last cell in any session that has the committed outputs of the others attached.
Submissions: `p7_out/submission_<run>.zip` (dense model only) and `submission_<run>_hybrid.zip` (dense + BM25); each holds `prediction.json` with exactly 7 unique candidate IDs per test query.

In [ ]:
# 1. Setup and switches
import os, re, ast, json, glob, math, time, random, zipfile, pickle, gc, collections, sys
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

P7_DRY_RUN = False                 # True: tiny end-to-end smoke test (~15 min). Set False for the real run.
P7_RUNS = [1, 2, 3, 4]             # runs this session executes (5/6/7 = optional ablations: in-batch negatives, margin loss, student without teacher)
P7_MINUTES = {1: 45, 2: 40, 3: 40, 4: 40, 5: 25, 6: 25, 7: 25}   # wall-clock training budget per run (LR schedule follows it)
P7_ZEROSHOT = True                 # evaluate the untouched Phase-1 models too (shows what the adaptation adds)
P7_SKIP_DONE = True                # skip a run whose submission and metrics already exist in the output folder
P7_TRUST_KNOWN = True              # reuse the pre-tokeniser / forward variants found in Phase 6 (verified by perplexity, searched again if wrong)
P7_STRATEGY = "select"             # long-case strategy: select | spread | headtail | lead
P7_MAX_CHUNKS = 4                  # chunks of <=512 tokens per case (2048 tokens of selected text)
P7_TOPK = 7                        # the task evaluates the top 7
P7_QBS = 16                        # queries per training step (each brings 1 positive + P7_HN hard negatives; the rest are in-batch negatives)
P7_HN = 3                          # hard negatives per query
P7_HN_POOL = 60                    # BM25 ranks (after skipping) the hard negatives are sampled from
P7_HN_SKIP = 2                     # skip the top BM25 non-relevant cases: labels are incomplete, the very top ones are often unlabeled positives
P7_TAU = 0.05                      # softmax temperature on cosine similarity
P7_MICRO = 16                      # sequences per forward/backward micro-batch (OOM auto-halves)
P7_ENC_BS = 48                     # sequences per inference batch (OOM auto-halves)
P7_MAX_EPOCHS = 12                 # upper bound in passes over the training queries (time budget usually ends first)
P7_SEED = 42
P7_OUT = "/kaggle/working/p7_out"
os.makedirs(P7_OUT, exist_ok=True)

random.seed(P7_SEED); np.random.seed(P7_SEED); torch.manual_seed(P7_SEED)
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU - attach a GPU accelerator")
print("DRY_RUN" if P7_DRY_RUN else "FULL RUN", "| runs this session:", P7_RUNS)

In [ ]:
# 2. Re-use the Phase-1 model code from the Phase-4 notebook (config, checkpoint discovery, tokenizer,
#    architectures 1-4, lenient loader) so the four Phase-1 models are rebuilt exactly as in Phases 4/5/6.
#    Attach phase4-allrun-v4.ipynb (as a dataset or via Add Input) and the four Phase-1 checkpoint datasets.
PHASE4_NOTEBOOK = None            # path to the Phase-4 .ipynb; None = search /kaggle/input and /kaggle/working
P4_BOOT_CELLS = (1, 2, 3, 4, 5, 6)
P4_REQUIRED = ("build_backbone", "pretokenize", "tokenizer", "lm_ppl", "ARCH_OPTS", "PREPROC_MODES")

def _p4_candidates():
    hits = glob.glob("/kaggle/input/**/*.ipynb", recursive=True) + glob.glob("/kaggle/working/**/*.ipynb", recursive=True)
    hits = [h for h in hits if not re.search(r"phase.?[567]|lsi|summ|pcr", os.path.basename(h), re.I)]
    hits.sort(key=lambda h: (not re.search(r"phase.?4|allrun|cjpe", h, re.I), not re.search(r"v4", h, re.I), len(h)))
    return hits

def _p4_cells(path):
    nb = json.load(open(path, encoding="utf-8")); out = {}
    for c in nb.get("cells", []):
        if c.get("cell_type") != "code": continue
        src = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        head = "\n".join(src.splitlines()[:5])
        m = re.search(r"^#\s*(?:CELL\s*)?(\d+)\.\s", head, re.M | re.I)
        if m and int(m.group(1)) not in out: out[int(m.group(1))] = src
    return out

def _sanitize(src):
    lines = []
    for l in src.splitlines():
        s = l.lstrip()
        lines.append(l[:len(l) - len(s)] + "pass  # [P7 skipped] " + s if s.startswith(("!", "%")) else l)
    return "\n".join(lines)

P4_ERRORS = []
if PHASE4_NOTEBOOK != "skip":
    _path = PHASE4_NOTEBOOK or next(iter(_p4_candidates()), None)
    assert _path, "Phase-4 notebook not found: attach phase4-allrun-v4.ipynb (Add Input) or set PHASE4_NOTEBOOK"
    print("Phase-4 notebook:", _path)
    _cells = _p4_cells(_path)
    print("numbered code cells found:", sorted(_cells))
    for _k in P4_BOOT_CELLS:
        if _k not in _cells: print(f"  cell {_k} missing"); continue
        try:
            exec(compile(_sanitize(_cells[_k]), f"<phase4 cell {_k}>", "exec"), globals())
        except Exception as e:
            P4_ERRORS.append((_k, repr(e))); print(f"  phase-4 cell {_k} raised: {repr(e)[:300]}")
    if 7 in _cells:   # cell 7 holds build_backbone / lm_ppl; take its definitions only, not the diagnosis run
        _tree = ast.parse(_sanitize(_cells[7])); _keep = []
        for _n in _tree.body:
            if isinstance(_n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "DIAG" for t in _n.targets): break
            if isinstance(_n, (ast.FunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom, ast.Assign)): _keep.append(_n)
        try:
            exec(compile(ast.Module(body=_keep, type_ignores=[]), "<phase4 cell 7 defs>", "exec"), globals())
        except Exception as e:
            P4_ERRORS.append((7, repr(e))); print("  phase-4 cell 7 definitions raised:", repr(e)[:300])
_missing = [n for n in P4_REQUIRED if n not in globals()]
assert not _missing, f"missing after import: {_missing}. Errors: {P4_ERRORS}. Paste Phase-4 cells 1-7 above and set PHASE4_NOTEBOOK='skip'."
device = globals().get("device", device)
P7_AMP_DTYPE = globals().get("AMP_DTYPE", torch.float16)
P7_AMP_ON = bool(globals().get("USE_AMP", True)) and device.type == "cuda"
print("checkpoints:", {a: bool(globals().get(f"ARCH{a}_CHECKPOINT")) for a in (1, 2, 3, 4)})

# diagnostics.json from a committed Phase-4/5/6 run (optional): saves the pre-tokeniser / variant search below
P4_DIAG = None
for _p in glob.glob("/kaggle/input/**/diagnostics.json", recursive=True) + glob.glob("/kaggle/working/**/diagnostics.json", recursive=True):
    try:
        _d = json.load(open(_p))
        if "preproc" in _d: P4_DIAG = _d; print("diagnostics.json:", _p, "| preproc =", _d["preproc"]); break
    except Exception: pass

In [ ]:
# 3. Phase-7 data: Train/Dev/Test queries and their own candidate pools
#    Phase_7_PCR/{Train,Dev,Test}/{train,dev,test}.json + {train,dev,test}_candidates.json
P7_USE_HF = True
P7_HF_REPO = None                      # "owner/name" if you know it; None = search the Exploration-Lab org (and "CS781") for a repo holding Phase_7 files
P7_HF_REPO_TYPE = "dataset"
P7_HF_ORGS = ["Exploration-Lab"]
P7_WAIT_MIN = 10                       # how long to wait for you to get access to a gated repo
P7_DATA_DIR = "/kaggle/working/p7_data"

def p7_hf_login():
    import getpass
    from huggingface_hub import login, whoami
    tok = os.environ.get("HF_TOKEN")
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN") or tok
    except Exception: pass
    if tok:
        try: login(token=tok, add_to_git_credential=False); print("logged in to Hugging Face as:", whoami(token=tok)["name"]); return tok
        except Exception as e: print("stored HF token rejected:", type(e).__name__)
    print("Hugging Face login: open https://huggingface.co/settings/tokens/new?tokenType=read (log in there), create a Read token, paste it below and press Enter.")
    tok = getpass.getpass("HF token: ").strip(); assert tok, "no token entered"
    login(token=tok, add_to_git_credential=False); print("logged in to Hugging Face as:", whoami(token=tok)["name"]); return tok

def p7_hf_discover(tok):
    """-> (repo_id, phase7_files, []) if an accessible repo holds Phase_7 json files, else (None, [], gated_repos_without_access)."""
    from huggingface_hub import HfApi
    from huggingface_hub.utils import GatedRepoError
    api = HfApi(token=tok); cands = [P7_HF_REPO] if P7_HF_REPO else []
    if not cands:
        for org in P7_HF_ORGS:
            try: cands += [d.id for d in api.list_datasets(author=org, limit=300)]
            except Exception as e: print("could not list", org, type(e).__name__)
        try: cands += [d.id for d in api.list_datasets(search="CS781", limit=50)]
        except Exception: pass
    gated = []
    for rid in dict.fromkeys(cands):
        try: files = api.list_repo_files(rid, repo_type=P7_HF_REPO_TYPE)
        except GatedRepoError: gated.append(rid); continue
        except Exception: continue
        want = [f for f in files if re.search(r"phase.?7", f, re.I) and f.lower().endswith((".json", ".jsonl"))]
        if want: return rid, want, []
    return None, [], gated

def p7_hf_fetch():
    from huggingface_hub import snapshot_download
    from IPython.display import display, HTML
    tok = p7_hf_login(); deadline = time.time() + P7_WAIT_MIN * 60; shown = False
    while True:
        rid, want, gated = p7_hf_discover(tok)
        if rid: break
        if not gated: raise FileNotFoundError("no accessible Hugging Face repo with Phase_7 files; set P7_HF_REPO = 'owner/name'")
        if not shown:
            links = "".join(f'<li><a href="https://huggingface.co/datasets/{g}" target="_blank">{g}</a></li>' for g in gated)
            display(HTML("<h3>Access needed</h3><p>Open the link(s), click <b>Agree / Request access</b> with the same account as your token. "
                         f"This cell re-checks every 20 s for {P7_WAIT_MIN} min.</p><ul>{links}</ul>")); shown = True
        if time.time() > deadline: raise TimeoutError("access to the gated repo was not granted in time")
        time.sleep(20)
    print("repo:", rid, "| Phase-7 files:", want)
    snapshot_download(rid, repo_type=P7_HF_REPO_TYPE, allow_patterns=want, local_dir=P7_DATA_DIR, token=tok)

def p7_find(name):
    c = sorted(glob.glob(f"{P7_DATA_DIR}/**/{name}", recursive=True)) + sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    pref = [p for p in c if re.search(r"phase.?7|pcr", p, re.I)]      # never fall back to another phase's train.json
    return (pref or [None])[0]

# ---------------- readers (tolerant of list / dict / jsonl layouts and sentence-list or string text) ----------------
_ID_KEYS = ("id", "doc_id", "case_id", "query_id", "ID", "document_id", "name")
_TXT_KEYS = ("text", "document", "judgment", "Document", "doc", "sentences", "content")
_REL_KEYS = ("relevant_candidates", "relevant", "relevant_cases", "labels", "gold", "citations")

def p7_read_json(path):
    txt = open(path, encoding="utf-8").read()
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

def p7_norm_text(t):
    if isinstance(t, (list, tuple)):
        out = []
        for s in t:
            if isinstance(s, (list, tuple)): s = " ".join(map(str, s))
            s = str(s).strip()
            if s: out.append(s)
        return out
    return [s.strip() for s in re.split(r"(?<=[.!?;])\s+|\n+", str(t)) if s and s.strip()]

def p7_rel_ids(rel):
    if isinstance(rel, dict): rel = list(rel.keys())
    out = []
    for x in rel:
        if isinstance(x, dict): x = next((x[k] for k in _ID_KEYS if k in x), None)
        if x is not None: out.append(str(x))
    return out

def p7_read_records(path):
    raw = p7_read_json(path)
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): raw = raw["data"]
        else: raw = [{"id": k, **(v if isinstance(v, dict) else {"text": v})} for k, v in raw.items()]
    recs = []
    for i, r in enumerate(raw):
        rid = next((r[k] for k in _ID_KEYS if k in r), i)
        txt = next((r[k] for k in _TXT_KEYS if k in r and r[k]), "")
        rel = next((r[k] for k in _REL_KEYS if k in r and r[k] is not None), None)
        recs.append(dict(id=str(rid), raw_id=rid, sents=p7_norm_text(txt), rel=None if rel is None else p7_rel_ids(rel)))
    return recs

def p7_dedupe(recs):
    seen, out = set(), []
    for r in recs:
        if r["id"] not in seen: seen.add(r["id"]); out.append(r)
    return out

def p7_dry_subset(Q, C, nq, extra, seed=0):
    rng = random.Random(seed); Q = Q[:nq]
    need = {r for q in Q for r in (q["rel"] or [])}
    keep = [c for c in C if c["id"] in need]; rest = [c for c in C if c["id"] not in need]
    rng.shuffle(rest); return Q, keep + rest[:extra]

print("cell 3: loading Phase-7 data ...")
P7_FILES = {s: (p7_find(f"{s}.json"), p7_find(f"{s}_candidates.json")) for s in ("train", "dev", "test")}
if P7_USE_HF and not all(all(v) for v in P7_FILES.values()):
    try: p7_hf_fetch()
    except Exception as e:
        print("Hugging Face download failed:", type(e).__name__, str(e)[:300])
        import traceback; traceback.print_exc()
        print("  If this is a 401/403/gated error: open the repo page while logged in, click Agree / Request access, and use a Read token from the same account. Internet must be ON. Falling back to /kaggle/input if the data is attached there.")
    P7_FILES = {s: (p7_find(f"{s}.json"), p7_find(f"{s}_candidates.json")) for s in ("train", "dev", "test")}
assert all(all(v) for v in P7_FILES.values()), f"Phase 7 data not found: {P7_FILES}. Fix the Hugging Face access above (or attach the data as a Kaggle dataset)."
for s, (a, b) in P7_FILES.items(): print(f"  {s:5s}: {a}\n         {b}")

P7_SPLITS = {}
for _s, (_qf, _cf) in P7_FILES.items():
    _Q, _C = p7_dedupe(p7_read_records(_qf)), p7_dedupe(p7_read_records(_cf))
    if _s != "test": assert all(q["rel"] is not None for q in _Q), f"{_s}: no relevance labels found in {_qf}; first record keys: {list(p7_read_json(_qf)[0].keys()) if isinstance(p7_read_json(_qf), list) else '?'}"
    if P7_DRY_RUN: _Q, _C = p7_dry_subset(_Q, _C, {"train": 48, "dev": 16, "test": 10}[_s], {"train": 200, "dev": 120, "test": 120}[_s])
    P7_SPLITS[_s] = (_Q, _C)
train_q, train_c = P7_SPLITS["train"]; dev_q, dev_c = P7_SPLITS["dev"]; test_q, test_c = P7_SPLITS["test"]

def p7_pool_check(name, Q, C):
    cid = {c["id"] for c in C}
    line = f"{name}: {len(Q)} queries | {len(C)} candidates"
    if Q and Q[0]["rel"] is not None:
        tot = sum(len(q["rel"]) for q in Q); inpool = sum(r in cid for q in Q for r in q["rel"])
        g = [len(q["rel"]) for q in Q]
        line += f" | gold/query mean {np.mean(g):.2f} median {int(np.median(g))} max {max(g)} | gold present in pool {inpool}/{tot}"
        if inpool < tot: print(f"  WARNING {name}: {tot - inpool} gold ids are not in the pool (ignored for training, kept in the metric denominator for Dev)")
    qself = sum(q["id"] in cid for q in Q)
    if qself: line += f" | {qself} query ids also appear in the pool (excluded from their own ranking)"
    print(line)
    qw = [sum(len(s.split()) for s in q["sents"]) for q in Q]; cw = [sum(len(s.split()) for s in c["sents"]) for c in C]
    print(f"   words p10/p50/p90  query {np.percentile(qw, [10, 50, 90]).astype(int).tolist()} | candidate {np.percentile(cw, [10, 50, 90]).astype(int).tolist()}")
for _n, (_q, _c) in P7_SPLITS.items(): p7_pool_check(_n, _q, _c)

# training keeps only gold that is in the pool; Dev/Test labels stay untouched
_tc_ids = {c["id"] for c in train_c}
for q in train_q: q["rel"] = [r for r in q["rel"] if r in _tc_ids]
dev_gold = {q["id"]: list(q["rel"]) for q in dev_q}
dev_qids, dev_cids = [q["id"] for q in dev_q], [c["id"] for c in dev_c]
test_qids, test_cids = [q["id"] for q in test_q], [c["id"] for c in test_c]
raw_test_q = {q["id"]: q["raw_id"] for q in test_q}; raw_test_c = {c["id"]: c["raw_id"] for c in test_c}
assert len(test_cids) >= P7_TOPK and len(dev_cids) >= P7_TOPK, "candidate pool smaller than 7"

In [ ]:
# 4. Text utilities: tokenisation, long-case sentence selection (same selector as Phase 6), chunk packing
def enc_text(t):
    return [int(i) for i in tokenizer.encode_words(pretokenize(t, PREPROC))]

_STOP = set("a an the of to in and or for on at by with from that this these those is are was were be been as it its which who whom whose he she they his her their them not no but if then than so such any all also may shall would should could can has have had do does did there here into under upon within without".split())
_CUES = re.compile(r"\b(held|dismiss|allow|appeal|convict|acquit|sentenc|conclude|therefore|accordingly|ordered|directed|quash|set aside|upheld|section|article|act)\w*", re.I)

def _words(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in _STOP and len(w) > 1]

def _sumbasic_pick(cand, wl, lens, budget, prob, cue):
    """Greedy SumBasic with redundancy update (probabilities of picked words are squared)."""
    prob = dict(prob); chosen = []; used = 0
    cand = list(cand)
    while cand:
        best, bs = None, -1.0
        for i in cand:
            w = wl[i]
            if not w or lens[i] > budget - used: continue
            sc = sum(prob.get(x, 0.0) for x in w) / len(w) + cue[i]
            if sc > bs: best, bs = i, sc
        if best is None: break
        chosen.append(best); used += lens[best]; cand.remove(best)
        for x in set(wl[best]): prob[x] = prob.get(x, 0.0) ** 2
    return chosen

def p7_select(sents, lens, budget, strategy="select"):
    """Indices (original order) of the sentences kept under a token budget."""
    n = len(sents)
    if n == 0: return []
    def take(rng, b):
        out, used = [], 0
        for i in rng:
            if used + lens[i] <= b: out.append(i); used += lens[i]
            elif used >= b * 0.9: break
        return out
    if sum(lens) <= budget: return list(range(n))
    if strategy == "lead": return take(range(n), budget)
    if strategy == "headtail":
        h = take(range(n), int(budget * 0.6)); hs = set(h)
        t = [i for i in take(range(n - 1, -1, -1), budget - sum(lens[i] for i in h)) if i not in hs]
        return sorted(set(h) | set(t))
    wl = [_words(s) for s in sents]; cnt = collections.Counter(w for x in wl for w in x); tot = max(1, sum(cnt.values()))
    prob = {w: c / tot for w, c in cnt.items()}
    cue = [0.002 * min(3, len(_CUES.findall(s))) for s in sents]
    init = sorted(range(n), key=lambda i: -(sum(prob.get(x, 0.0) for x in wl[i]) / max(1, len(wl[i]))))[:250]
    if strategy == "spread":
        S = 6; chosen = []; seg = [(k * n // S, (k + 1) * n // S) for k in range(S)]
        for a, b in seg:
            pool = sorted(range(a, b), key=lambda i: -(sum(prob.get(x, 0.0) for x in wl[i]) / max(1, len(wl[i]))))[:60]
            chosen += _sumbasic_pick(pool, wl, lens, budget // S, prob, cue)
        return sorted(set(chosen))
    # "select": lead + tail (judgments end with the operative order) + SumBasic over the middle
    lead = take(range(n), int(budget * 0.25)); ls = set(lead)
    tail = [i for i in take(range(n - 1, -1, -1), int(budget * 0.25)) if i not in ls]; ts = set(tail)
    rest = budget - sum(lens[i] for i in lead) - sum(lens[i] for i in tail)
    mid = _sumbasic_pick([i for i in init if i not in ls and i not in ts], wl, lens, rest, prob, cue)
    return sorted(set(lead) | set(tail) | set(mid))

P7_SENT_CAP = 96   # tokens kept per sentence while selecting
def p7_pack_chunks(sents, enc, strategy, n_chunks, chunk_len, pad=0):
    """Select sentences under a budget, then pack them (original order) into at most n_chunks sentence-aligned chunks of <= chunk_len tokens."""
    ids = [(s, enc(s)[:P7_SENT_CAP]) for s in sents if s and s.strip()]
    ids = [(s, t) for s, t in ids if t]
    if not ids: return [np.array([pad], dtype=np.int32)]
    ss, tt = [s for s, _ in ids], [t for _, t in ids]
    keep = p7_select(ss, [len(t) for t in tt], int(n_chunks * chunk_len * 0.92), strategy)
    chunks, cur = [], []
    for i in keep:
        t = tt[i]
        if cur and len(cur) + len(t) > chunk_len:
            chunks.append(cur); cur = []
            if len(chunks) >= n_chunks: break
        cur.extend(t)
    if cur and len(chunks) < n_chunks: chunks.append(cur)
    return [np.asarray(c, dtype=np.int32) for c in chunks] or [np.array([pad], dtype=np.int32)]

In [ ]:
# 5. Sequence layout, pre-tokeniser and forward-variant check on Phase-7 text
#    Phase 6 found: pre-tokeniser "split"; Arch3 {kv: interleave, act: gelu}; Arch2 {rope: half, qkn: after, swap: False}; Arch1 {act: gelu}.
#    With P7_TRUST_KNOWN these are applied and verified by perplexity; if a model is far off, the grid search runs again for it.
PAD = int(globals().get("PAD_ID", 0))
CTX = min(512, int(globals().get("MAX_POS", 512)))          # tokens per chunk
P7_KNOWN_OPTS = {3: {"kv": "interleave", "act": "gelu"}, 2: {"rope": "half", "qkn": "after", "swap": False}, 1: {"act": "gelu"}}
VARIANT_GRID = {
    3: [{"kv": kv, "act": a} for kv in ("interleave", "repeat") for a in ("gelu", "gelu_tanh", "relu")],
    2: [{"rope": r, "qkn": q, "swap": s} for r in ("half", "pairs") for q in ("before", "after") for s in (False, True)],
    1: [{"act": a} for a in ("gelu", "gelu_tanh", "relu")],
}

def p7_free():
    gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()

def p7_ppl_sample(mode, n=24, seed=0):
    rng = random.Random(seed)
    for L in (256, 128, 64):
        rows = []
        for r in rng.sample(train_c, min(400, len(train_c))):
            w = pretokenize(" ".join(r["sents"]), mode)
            if len(w) < L * 2: continue
            mid = len(w) // 2; ids = tokenizer.encode_words(w[mid:mid + L + L // 2])[:L]
            if len(ids) == L: rows.append(ids)
            if len(rows) >= n: break
        if rows: return torch.tensor(rows, dtype=torch.long)
    raise RuntimeError("cases too short for the perplexity check")

if P4_DIAG and "preproc" in P4_DIAG: PREPROC = P4_DIAG["preproc"]
if PREPROC == "auto":
    if P7_TRUST_KNOWN: PREPROC = "split"
    else:
        _smp = {m: p7_ppl_sample(m) for m in PREPROC_MODES}; _tab = {}
        for a in (4, 1):
            bb = build_backbone(a, f"Arch{a} (pre-tokeniser check)")
            _tab[a] = {m: lm_ppl(bb, _smp[m]) for m in PREPROC_MODES}; del bb; p7_free()
        PREPROC = min(PREPROC_MODES, key=lambda m: sum(math.log(_tab[a][m]) for a in _tab))
        print("pre-tokeniser table:", {a: {m: round(v, 1) for m, v in d.items()} for a, d in _tab.items()}, "->", PREPROC)
PPL_IDS7 = p7_ppl_sample(PREPROC)
_saved_opts = (P4_DIAG or {}).get("arch_opts") or (P4_DIAG or {}).get("variants") or {}
P7_PPL = {}
for a in (4, 3, 2, 1):
    if not globals().get(f"ARCH{a}_CHECKPOINT"): print(f"Arch{a}: checkpoint missing"); continue
    known = False
    if str(a) in _saved_opts and a in VARIANT_GRID: ARCH_OPTS[a] = dict(_saved_opts[str(a)]); known = True
    elif P7_TRUST_KNOWN and a in P7_KNOWN_OPTS: ARCH_OPTS[a] = dict(P7_KNOWN_OPTS[a]); known = True
    bb = build_backbone(a, f"Arch{a} (variant check)")
    ppl = lm_ppl(bb, PPL_IDS7)
    if a in VARIANT_GRID and (not known or ppl > 1500):
        res = []
        for opt in VARIANT_GRID[a]:
            ARCH_OPTS[a] = dict(opt); res.append((lm_ppl(bb, PPL_IDS7), opt))
        res.sort(key=lambda r: r[0]); ARCH_OPTS[a] = dict(res[0][1]); ppl = res[0][0]
    P7_PPL[a] = ppl; print(f"Arch{a}: Phase-7 text perplexity {ppl:.1f} {ARCH_OPTS.get(a, '')}")
    del bb; p7_free()
print("PREPROC =", PREPROC, "| chunk length", CTX, "| chunks per case", P7_MAX_CHUNKS)

In [ ]:
# 6. Retrieval utilities (no neural weights): metrics, top-k, row z-scores, BM25 baseline
from scipy import sparse

def p7_micro_prf(pred, gold, k=7):
    """Micro precision / recall / F1 @k. gold is neither truncated nor padded; only the first k predictions count."""
    tp = npred = ngold = 0
    for q, g in gold.items():
        p = list(pred.get(q, []))[:k]; g = set(g)
        tp += len(set(p) & g); npred += len(p); ngold += len(g)
    P, R = tp / max(1, npred), tp / max(1, ngold)
    return P, R, (2 * P * R / (P + R) if P + R > 0 else 0.0)

def p7_topk(S, qids, cids, k=7):
    """Rank candidates by score; a case never retrieves itself. -> {query_id: [k candidate ids]}."""
    S = np.nan_to_num(np.array(S, dtype=np.float32), nan=-1e9, posinf=1e9, neginf=-1e9)
    pos = {c: i for i, c in enumerate(cids)}
    for qi, q in enumerate(qids):
        j = pos.get(q)
        if j is not None: S[qi, j] = -np.inf
    k = min(k, S.shape[1])
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    out = {}
    for qi, q in enumerate(qids):
        o = idx[qi][np.argsort(-S[qi, idx[qi]], kind="stable")]
        out[q] = [cids[j] for j in o]
    return out

def p7_z(S):
    S = np.asarray(S, dtype=np.float32); return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-8)

def p7_case_words(rec): return _words(" ".join(rec["sents"]))

class P7BM25:
    """Okapi BM25 over unigram terms (stop words removed); long queries use 1+log(query tf) term weights."""
    def __init__(self, docs_words):
        vocab = {}; indptr = [0]; indices = []; data = []
        for toks in docs_words:
            for w, tf in collections.Counter(toks).items():
                indices.append(vocab.setdefault(w, len(vocab))); data.append(tf)
            indptr.append(len(indices))
        self.vocab = vocab; n = len(docs_words)
        self.tf = sparse.csr_matrix((np.asarray(data, np.float32), np.asarray(indices, np.int32), np.asarray(indptr, np.int64)), shape=(n, max(1, len(vocab))))
        self.dl = np.asarray([len(t) for t in docs_words], np.float32); self.avg = max(1.0, float(self.dl.mean()))
        df = np.bincount(self.tf.indices, minlength=self.tf.shape[1]).astype(np.float32)
        self.idf = np.log(1.0 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        self.rows = np.repeat(np.arange(n), np.diff(self.tf.indptr))
    def weights(self, k1=1.2, b=0.75):
        tf = self.tf.data; cols = self.tf.indices
        w = tf * (k1 + 1.0) / (tf + k1 * (1.0 - b + b * self.dl[self.rows] / self.avg)) * self.idf[cols]
        return sparse.csr_matrix((w.astype(np.float32), cols, self.tf.indptr), shape=self.tf.shape)
    def query_matrix(self, qwords):
        indptr = [0]; indices = []; data = []
        for toks in qwords:
            for w, c in collections.Counter(toks).items():
                j = self.vocab.get(w)
                if j is not None: indices.append(j); data.append(1.0 + math.log(c))
            indptr.append(len(indices))
        return sparse.csr_matrix((np.asarray(data, np.float32), np.asarray(indices, np.int32), np.asarray(indptr, np.int64)), shape=(len(qwords), self.tf.shape[1]))
    def scores(self, qwords, k1=1.2, b=0.75, block=256):
        W = self.weights(k1, b).T.tocsc(); out = []
        for s in range(0, len(qwords), block):
            out.append((self.query_matrix(qwords[s:s + block]) @ W).toarray().astype(np.float32))
        return np.concatenate(out) if out else np.zeros((0, self.tf.shape[0]), np.float32)

In [ ]:
# 7. BM25 baseline (Dev/Test), BM25 hard negatives for training, chunk caches
#    BM25 uses no pretrained weights; it is the reference every dense run is compared with, the source of hard negatives,
#    and the lexical half of the hybrid submissions.
_tag = "dry" if P7_DRY_RUN else "full"
t0 = time.time()
dev_cw = [p7_case_words(c) for c in dev_c]; dev_qw = [p7_case_words(q) for q in dev_q]
bm_dev = P7BM25(dev_cw)
P7_BM25_GRID = [(1.2, 0.75), (0.9, 0.4), (1.5, 0.75), (2.0, 0.75), (1.2, 0.9), (1.2, 0.5)]
BM25_DEV_TABLE, _best = [], None
for _k1, _b in P7_BM25_GRID:
    _S = bm_dev.scores(dev_qw, _k1, _b); _P, _R, _F = p7_micro_prf(p7_topk(_S, dev_qids, dev_cids, P7_TOPK), dev_gold, P7_TOPK)
    BM25_DEV_TABLE.append(dict(k1=_k1, b=_b, P=round(_P, 4), R=round(_R, 4), F1=round(_F, 4)))
    if _best is None or _F > _best[0]: _best = (_F, (_k1, _b), _S)
BM25_PARAMS, S_bm_dev = _best[1], _best[2]
print("BM25 on Dev (micro-F1@7):", {f"k1={r['k1']},b={r['b']}": r["F1"] for r in BM25_DEV_TABLE}, "-> best", BM25_PARAMS, f"| {time.time()-t0:.0f}s")
test_cw = [p7_case_words(c) for c in test_c]; test_qw = [p7_case_words(q) for q in test_q]
S_bm_test = P7BM25(test_cw).scores(test_qw, *BM25_PARAMS)
np.save(f"{P7_OUT}/bm25_dev_{_tag}.npy", S_bm_dev); np.save(f"{P7_OUT}/bm25_test_{_tag}.npy", S_bm_test)
BM25_DEV = dict(zip(("P", "R", "F1"), p7_micro_prf(p7_topk(S_bm_dev, dev_qids, dev_cids, P7_TOPK), dev_gold, P7_TOPK)))
print("BM25 Dev P/R/F1 @7:", {k: round(v, 4) for k, v in BM25_DEV.items()})

# ---------------- BM25 hard negatives for the training queries ----------------
t0 = time.time()
tr_cw = [p7_case_words(c) for c in train_c]; tr_qw = [p7_case_words(q) for q in train_q]
bm_tr = P7BM25(tr_cw); _W = bm_tr.weights(*BM25_PARAMS).T.tocsc()
t_cidx = {c["id"]: i for i, c in enumerate(train_c)}
T_POS = [np.asarray([t_cidx[r] for r in q["rel"]], dtype=np.int64) for q in train_q]
T_HN, _tp, _np_ = [], 0, 0
for _s in range(0, len(train_q), 128):
    _S = (bm_tr.query_matrix(tr_qw[_s:_s + 128]) @ _W).toarray()
    for _i in range(_S.shape[0]):
        qi = _s + _i; row = _S[_i]; me = t_cidx.get(train_q[qi]["id"]); ps = set(T_POS[qi].tolist())
        m = min(len(row) - 1, P7_HN_POOL + P7_HN_SKIP + len(ps) + 2)
        top = np.argpartition(-row, m)[:m + 1]; top = top[np.argsort(-row[top])]
        _tp += len(set(top[:P7_TOPK].tolist()) & ps); _np_ += min(P7_TOPK, len(top))
        neg = [int(j) for j in top if int(j) not in ps and j != me]
        T_HN.append(np.asarray(neg[P7_HN_SKIP:P7_HN_SKIP + P7_HN_POOL] or neg[:P7_HN_POOL], dtype=np.int64))
print(f"BM25 hard negatives mined for {len(train_q)} train queries | train precision@7 of BM25 {_tp / max(1, _np_):.3f} | {time.time() - t0:.0f}s")

# ---------------- chunk caches (token ids of the selected, chunked cases) ----------------
def p7_cache_chunks(recs, tag, only=None):
    idxs = list(range(len(recs))) if only is None else sorted(only)
    f = f"{P7_OUT}/chunks_{tag}_{PREPROC}_{P7_STRATEGY}_{P7_MAX_CHUNKS}_{CTX}_{len(recs)}_{len(idxs)}_{_tag}.pkl"
    if os.path.exists(f):
        try: return pickle.load(open(f, "rb"))
        except Exception: pass
    t0 = time.time(); out = {}
    for n, i in enumerate(idxs):
        out[i] = p7_pack_chunks(recs[i]["sents"], enc_text, P7_STRATEGY, P7_MAX_CHUNKS, CTX, PAD)
        if n and n % 1000 == 0: print(f"   chunks {tag}: {n}/{len(idxs)}  {time.time()-t0:.0f}s")
    pickle.dump(out, open(f, "wb")); print(f"chunks {tag}: {len(idxs)} cases in {time.time()-t0:.0f}s"); return out

def p7_chunk_list(d, n): return [d[i] for i in range(n)]

DEV_Q_CH = p7_chunk_list(p7_cache_chunks(dev_q, "devq"), len(dev_q)); DEV_C_CH = p7_chunk_list(p7_cache_chunks(dev_c, "devc"), len(dev_c))
TEST_Q_CH = p7_chunk_list(p7_cache_chunks(test_q, "testq"), len(test_q)); TEST_C_CH = p7_chunk_list(p7_cache_chunks(test_c, "testc"), len(test_c))
TQ_CH = p7_chunk_list(p7_cache_chunks(train_q, "trainq"), len(train_q))
_need = set(int(j) for p in T_POS for j in p) | set(int(j) for h in T_HN for j in h)
TC_CH = p7_cache_chunks(train_c, "trainc", only=_need)
T_NEED = sorted(TC_CH)
T_VALID = [i for i in range(len(train_q)) if len(T_POS[i]) > 0]
_nch = [len(c) for c in DEV_C_CH]
print(f"train queries usable {len(T_VALID)} | train candidates chunked {len(TC_CH)}/{len(train_c)} | dev pool chunks/case mean {np.mean(_nch):.2f} | total dev+test chunks to encode {sum(map(len, DEV_Q_CH)) + sum(map(len, DEV_C_CH)) + sum(map(len, TEST_Q_CH)) + sum(map(len, TEST_C_CH))}")

In [ ]:
# 8. Adaptation techniques (all start from the Phase-1 weights) and the retrieval encoder
#    LoRMA   : multiplicative low-rank update  W' = (I + B A) W   (B = 0 at start -> identical to Phase 1)
#    LoRA    : additive low-rank update        W' = W + B A       (not a required run, kept for ablations)
#    IA3     : learned rescaling of attention keys / values (outputs) and of the FFN hidden activations (input of the down projection)
#    Partial : only the last N blocks and the final norm are trained
#    KD      : full fine-tuning of a half-depth student against a stronger teacher (see the training cell)
#    Training runs with dropout off (eval mode) so that the GradCache forward passes agree with each other.
class P7LoRALinear(nn.Module):
    def __init__(self, base, r=16, alpha=32, mode="lora"):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        d_in = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, d_in)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); z = x if self.mode == "lora" else y     # lorma: low-rank map of the layer OUTPUT (multiplicative)
        return y + self.scale * F.linear(F.linear(z, self.A), self.B)

class P7ScaleOut(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x):
        y = self.base(x); return y * self.l.to(y.dtype)

class P7ScaleIn(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x): return self.base(x * self.l.to(x.dtype))

def p7_set_module(root, dotted, new):
    parent = root; parts = dotted.split(".")
    for p in parts[:-1]: parent = getattr(parent, p)
    setattr(parent, parts[-1], new)

def p7_replace_linears(root, make):
    for name, m in [(n, m) for n, m in root.named_modules() if isinstance(m, nn.Linear)]:
        p7_set_module(root, name, make(m))

_ATT = {"attn", "attention", "self_attn", "self_attention"}
_FFN = {"ffn", "mlp", "feed_forward", "ff"}
_K = {"k", "key", "k_proj", "wk", "key_proj", "w_k"}
_V = {"v", "value", "v_proj", "wv", "value_proj", "w_v"}
_QKV = {"qkv", "c_attn", "in_proj", "wqkv", "kv", "qkv_proj", "kv_proj"}

def p7_add_ia3(bb):
    """(IA)^3: rescale keys and values (attention) and the FFN hidden activations. Which Linear is which is decided by name, then by shape."""
    n_att = n_ffn = 0; shown = False
    for blk in bb.blocks:
        for cname, child in list(blk.named_children()):
            lc = cname.lower()
            if lc in _ATT:
                lins = [(n, m) for n, m in child.named_modules() if isinstance(m, nn.Linear)]
                pick = [n for n, _ in lins if n.split(".")[-1].lower() in (_K | _V | _QKV)]
                if not pick: pick = [n for n, _ in lins[:-1]]            # unknown naming: all but the last (assumed output projection)
                for n in pick:
                    p7_set_module(child, n, P7ScaleOut(dict(lins)[n])); n_att += 1
            elif lc in _FFN:
                lins = [(n, m) for n, m in child.named_modules() if isinstance(m, nn.Linear)]
                down = [n for n, m in lins if m.in_features > m.out_features]
                if not down: down = [lins[-1][0]]
                p7_set_module(child, down[-1], P7ScaleIn(dict(lins)[down[-1]])); n_ffn += 1
        if not shown:
            shown = True; print("  IA3 block-0 modules:", [n for n, m in blk.named_modules() if isinstance(m, (P7ScaleOut, P7ScaleIn))])
    assert n_att and n_ffn, f"IA3: no attention/FFN Linear found; block children: {[c for c, _ in bb.blocks[0].named_children()]}"
    print(f"  IA3 scaling vectors: {n_att} attention (K/V) + {n_ffn} FFN")

_FINAL_NORMS = {"norm", "ln_f", "final_norm", "ln_final", "norm_f", "final_ln", "ln_out"}
def p7_unfreeze_last(bb, n_last):
    for p in bb.parameters(): p.requires_grad = False
    for blk in list(bb.blocks)[-n_last:]:
        for p in blk.parameters(): p.requires_grad = True
    for cname, child in bb.named_children():
        if cname.lower() in _FINAL_NORMS:
            for p in child.parameters(): p.requires_grad = True

class P7Enc(nn.Module):
    """Right-padded batch -> (mean-pooled, last-token) hidden states. A causal model never looks at tokens to its right, so pads cannot leak in."""
    def __init__(self, bb): super().__init__(); self.bb = bb
    def hidden(self, ids):
        h = self.bb(ids); return h[0] if isinstance(h, (tuple, list)) else h
    def forward(self, ids, lens):
        h = self.hidden(ids).float(); B, T = ids.shape
        m = (torch.arange(T, device=ids.device)[None, :] < lens[:, None]).float()
        mean = (h * m[..., None]).sum(1) / lens.clamp(min=1)[:, None].float()
        last = h[torch.arange(B, device=ids.device), (lens - 1).clamp(min=0)]
        return mean, last

def p7_make_model(arch, tech, cfg, name):
    bb = build_backbone(arch, name)
    if cfg.get("keep"):                                   # half-depth student
        n_keep = max(1, int(round(len(bb.blocks) * cfg["keep"]))); bb.blocks = nn.ModuleList(list(bb.blocks)[:n_keep])
        print(f"  {name}: kept the first {n_keep} blocks")
    for p in bb.parameters(): p.requires_grad = False
    if tech in ("kd", "full"):
        for p in bb.parameters(): p.requires_grad = True
    elif tech == "partial": p7_unfreeze_last(bb, cfg.get("n_last", 4))
    elif tech in ("lora", "lorma"):
        r = cfg.get("r", 16); p7_replace_linears(bb.blocks, lambda m: P7LoRALinear(m, r, 2 * r, tech))
    elif tech == "ia3": p7_add_ia3(bb)
    elif tech != "none": raise ValueError(tech)
    return P7Enc(bb).to(device).eval()

def p7_count(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)

def p7_pad(seqs):
    L = max(len(s) for s in seqs); ids = np.full((len(seqs), L), PAD, dtype=np.int64)
    for i, s in enumerate(seqs): ids[i, :len(s)] = s
    return torch.from_numpy(ids), torch.tensor([len(s) for s in seqs], dtype=torch.long)

def p7_embed_seqs(model, seqs, pool, micro, amp, grad):
    """Raw (un-normalised) pooled vectors for a list of token arrays."""
    outs = []
    for s in range(0, len(seqs), micro):
        ids, lens = p7_pad(seqs[s:s + micro])
        with torch.set_grad_enabled(grad), torch.autocast(device_type=device.type, dtype=P7_AMP_DTYPE, enabled=amp):
            mean, last = model(ids.to(device), lens.to(device))
        outs.append(mean if pool == "mean" else last)
    return torch.cat(outs)

def p7_amp_ok(model, seqs):
    """Mixed precision is kept only if it reproduces the fp32 embeddings."""
    if not P7_AMP_ON: return False
    with torch.no_grad():
        a = F.normalize(p7_embed_seqs(model, seqs, "mean", 8, False, False), dim=-1)
        b = F.normalize(p7_embed_seqs(model, seqs, "mean", 8, True, False), dim=-1)
    ok = bool(torch.isfinite(b).all()) and float((a * b).sum(-1).min()) > 0.995
    print(f"  amp check: min cosine fp32 vs amp {float((a * b).sum(-1).min()):.5f} -> {'amp on' if ok else 'fp32 (amp unsafe)'}"); return ok

In [ ]:
# 9. Encoding, aggregation of chunk vectors into case vectors, similarity, Dev evaluation of every pooling/aggregation combination
#    pool  : mean (over tokens) | last (token)
#    centre: subtract the mean vector of the split (queries + pool) before normalising (fights the anisotropy of LM states)
#    agg   : first chunk | mean of chunks | MaxSim (mean over query chunks of the best candidate chunk) | max-max
P7_POOLS = ("mean", "last"); P7_AGGS = ("first", "mean", "meanmax", "maxmax")

@torch.no_grad()
def p7_encode(model, chunks_docs, amp, bs=None):
    """chunks_docs: list over cases of lists of token arrays -> dict of L2-normalised chunk vectors (fp16) with their case / chunk index."""
    flat, doc, pos = [], [], []
    for di, chs in enumerate(chunks_docs):
        for ci, c in enumerate(chs): flat.append(c); doc.append(di); pos.append(ci)
    order = np.argsort([len(c) for c in flat])[::-1]; model.eval()
    Em = El = None; i = 0; b = bs or P7_ENC_BS; t0 = time.time()
    while i < len(order):
        idx = order[i:i + b]
        try:
            ids, lens = p7_pad([flat[j] for j in idx])
            with torch.autocast(device_type=device.type, dtype=P7_AMP_DTYPE, enabled=amp):
                mean, last = model(ids.to(device), lens.to(device))
        except torch.cuda.OutOfMemoryError:
            p7_free()
            if b == 1: raise
            b = max(1, b // 2); print(f"   OOM -> encode batch {b}"); continue
        if Em is None: d = mean.size(1); Em = np.zeros((len(flat), d), np.float16); El = np.zeros((len(flat), d), np.float16)
        Em[idx] = F.normalize(mean, dim=-1).half().cpu().numpy(); El[idx] = F.normalize(last, dim=-1).half().cpu().numpy(); i += len(idx)
    if device.type == "cuda": torch.cuda.synchronize()
    sec = time.time() - t0
    if not (np.isfinite(Em).all() and np.isfinite(El).all()): print("   WARNING: non-finite embeddings (fp16 overflow?)"); Em = np.nan_to_num(Em); El = np.nan_to_num(El)
    return dict(mean=Em, last=El, doc=np.asarray(doc), pos=np.asarray(pos), sec=sec, n=len(flat), ndocs=len(chunks_docs))

def p7_pack(E, pool, mu=None):
    X = E[pool].astype(np.float32)
    if mu is not None: X = X - mu; X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    C = int(E["pos"].max()) + 1; N = E["ndocs"]
    P = np.zeros((N, C, X.shape[1]), np.float32); M = np.zeros((N, C), bool)
    P[E["doc"], E["pos"]] = X; M[E["doc"], E["pos"]] = True
    return torch.from_numpy(P), torch.from_numpy(M)

def p7_scores(Qp, Qm, Dp, Dm, agg):
    Qp, Qm, Dp, Dm = [t.to(device) for t in (Qp, Qm, Dp, Dm)]
    def meanpool(P, M): return F.normalize((P * M[..., None]).sum(1) / M.sum(1, keepdim=True).clamp(min=1), dim=-1)
    if agg == "first": return (F.normalize(Qp[:, 0], dim=-1) @ F.normalize(Dp[:, 0], dim=-1).T).cpu().numpy()
    if agg == "mean": return (meanpool(Qp, Qm) @ meanpool(Dp, Dm).T).cpu().numpy()
    Nq, Cq, _ = Qp.shape; Nd, Cd, _ = Dp.shape; blk = max(1, int(4e7 // max(1, Nd * Cq * Cd))); out = []
    for s in range(0, Nq, blk):
        sim = torch.einsum("bqd,ncd->bnqc", Qp[s:s + blk], Dp).masked_fill(~Dm[None, :, None, :], -2.0)
        mx = sim.max(-1).values; qm = Qm[s:s + blk][:, None, :]
        v = (mx * qm).sum(-1) / qm.sum(-1).clamp(min=1) if agg == "meanmax" else mx.masked_fill(~qm, -2.0).max(-1).values
        out.append(v.cpu())
    return torch.cat(out).numpy()

def p7_combo_scores(Eq, Ed, key):
    pool, cen, agg = key.split("|")
    mu = np.concatenate([Eq[pool], Ed[pool]]).astype(np.float32).mean(0) if cen == "center" else None
    Qp, Qm = p7_pack(Eq, pool, mu); Dp, Dm = p7_pack(Ed, pool, mu)
    return p7_scores(Qp, Qm, Dp, Dm, agg)

def p7_eval_combos(Eq, Ed, qids, cids, gold, k=7):
    """Every pool x centre x aggregation combination on Dev -> (table, best_key, best_scores)."""
    res, best = {}, None
    for pool in P7_POOLS:
        mu = np.concatenate([Eq[pool], Ed[pool]]).astype(np.float32).mean(0)
        for cen in ("raw", "center"):
            Qp, Qm = p7_pack(Eq, pool, mu if cen == "center" else None); Dp, Dm = p7_pack(Ed, pool, mu if cen == "center" else None)
            for agg in P7_AGGS:
                S = p7_scores(Qp, Qm, Dp, Dm, agg); P, R, F1 = p7_micro_prf(p7_topk(S, qids, cids, k), gold, k)
                key = f"{pool}|{cen}|{agg}"; res[key] = dict(P=round(P, 4), R=round(R, 4), F1=round(F1, 4))
                if best is None or F1 > best[1]: best = (key, F1, S)
    return res, best[0], best[2]

def p7_hybrid_tune(Sd, Sb, qids, cids, gold, k=7, lams=(0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0)):
    """z-scored dense + lambda * z-scored BM25; lambda picked on Dev."""
    zd, zb = p7_z(Sd), p7_z(Sb); best = (0.0, -1.0, None); table = {}
    for lam in lams:
        P, R, F1 = p7_micro_prf(p7_topk(zd + lam * zb, qids, cids, k), gold, k); table[lam] = round(F1, 4)
        if F1 > best[1]: best = (lam, F1, (P, R))
    return best[0], best[1], best[2], table

def p7_write_submission(path, preds, raw_q, raw_c, qids, cid_set):
    """prediction.json inside a zip: exactly 7 unique ids per test query, all from the test pool, best first."""
    out = {}
    for q in qids:
        ids = preds[q]; assert len(ids) == P7_TOPK and len(set(ids)) == P7_TOPK and all(i in cid_set for i in ids), f"bad prediction for {q}"
        out[raw_q[q]] = [raw_c[i] for i in ids]
    assert len(out) == len(qids)
    tmp = path.replace(".zip", "_prediction.json"); json.dump(out, open(tmp, "w"))
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z: z.write(tmp, "prediction.json")
    os.remove(tmp)

In [ ]:
# 10. Training: chunk-level contrastive learning with GradCache, losses (InfoNCE / margin / distillation), time-budgeted loop
#     A step = B queries, each with 1 sampled positive and H negatives (hard = BM25, random, or none). Every document in the batch is a
#     negative for every query unless it is a known positive of that query (masked out, because the labels are incomplete).
#     GradCache: (1) embed everything without grad, (2) compute the loss on the cached vectors and get d(loss)/d(vector),
#     (3) re-run small micro-batches with grad and push those gradients through. The negatives pool is large, the memory is not.
def p7_sample(rng, B, H, neg):
    qs = rng.sample(T_VALID, min(B, len(T_VALID))); q_seqs = []; docs = []; col = {}; pos_col = []; posset = []
    def add(c):
        if c not in col: col[c] = len(docs); docs.append(c)
        return col[c]
    for qi in qs:
        ps = set(int(x) for x in T_POS[qi]); posset.append(ps)
        q_seqs.append(TQ_CH[qi][rng.randrange(len(TQ_CH[qi]))])
        pos_col.append(add(int(rng.choice(list(ps)))))
        for _ in range(H if neg != "inbatch" else 0):
            if neg == "hard" and len(T_HN[qi]): n = int(T_HN[qi][rng.randrange(len(T_HN[qi]))])
            else:
                n = T_NEED[rng.randrange(len(T_NEED))]
                for _try in range(10):
                    if n not in ps: break
                    n = T_NEED[rng.randrange(len(T_NEED))]
            if n not in ps: add(n)
    mask = np.zeros((len(qs), len(docs)), bool)
    for i, ps in enumerate(posset):
        for j, c in enumerate(docs):
            if c in ps and j != pos_col[i]: mask[i, j] = True
    d_seqs = [TC_CH[c][rng.randrange(len(TC_CH[c]))] for c in docs]
    return dict(q=q_seqs, d=d_seqs, pos=np.asarray(pos_col), mask=mask)

def p7_loss(q, d, pos, mask, cfg, tq=None, td=None):
    """q,d raw vectors [B,dim],[N,dim]; pos [B] column of each query's positive; mask [B,N] True = known positive that must not count as a negative."""
    tau = cfg.get("tau", P7_TAU); qn, dn = F.normalize(q, dim=-1), F.normalize(d, dim=-1)
    S = qn @ dn.T; B, N = S.shape; ar = torch.arange(B, device=S.device)
    L = (S / tau).masked_fill(mask, -1e4); kind = cfg.get("loss", "infonce")
    if kind == "margin":                                     # pairwise: positive must beat the hardest negative by a margin
        onehot = F.one_hot(pos, N).bool(); hard = S.masked_fill(mask | onehot, -2.0).max(1).values
        main = F.relu(cfg.get("margin", 0.2) - S[ar, pos] + hard).mean()
    else:                                                    # listwise InfoNCE, symmetrised with positive -> query
        ce1 = F.cross_entropy(L, pos)
        m2 = mask[:, pos].T | (pos[:, None] == pos[None, :]); m2 = m2 & ~torch.eye(B, dtype=torch.bool, device=S.device)
        ce2 = F.cross_entropy((dn[pos] @ qn.T / tau).masked_fill(m2, -1e4), ar); main = 0.5 * (ce1 + ce2)
    stats = dict(acc=float((L.argmax(1) == pos).float().mean()))
    if tq is None: return main, stats
    T = cfg.get("T", 2.0); tqn, tdn = F.normalize(tq, dim=-1), F.normalize(td, dim=-1)
    TL = (tqn @ tdn.T / tau).masked_fill(mask, -1e4)
    kd = F.kl_div(F.log_softmax(L / T, -1), F.softmax(TL / T, -1), reduction="batchmean") * T * T          # teacher score distribution
    emb = 0.5 * ((1 - F.cosine_similarity(q, tq, dim=-1)).mean() + (1 - F.cosine_similarity(d, td, dim=-1)).mean())   # intermediate representation
    a = cfg.get("alpha", 0.5); stats.update(kd=float(kd), emb=float(emb))
    return a * main + (1 - a) * kd + cfg.get("beta", 0.5) * emb, stats

def p7_train_step(model, teacher, cfg, batch, micro, amp, scaler):
    pool = cfg.get("pool", "mean"); seqs = batch["q"] + batch["d"]; B = len(batch["q"]); dev_ = device
    with torch.no_grad(): E = p7_embed_seqs(model, seqs, pool, micro, amp, False).float()
    E = E.detach().requires_grad_(True)
    tq = td = None
    if teacher is not None:
        with torch.no_grad(): TE = p7_embed_seqs(teacher, seqs, pool, micro, amp, False).float()
        tq, td = TE[:B], TE[B:]
    loss, stats = p7_loss(E[:B], E[B:], torch.as_tensor(batch["pos"], device=dev_), torch.as_tensor(batch["mask"], device=dev_), cfg, tq, td)
    loss.backward(); G = E.grad.detach()
    for i in range(0, len(seqs), micro):
        part = seqs[i:i + micro]; e = p7_embed_seqs(model, part, pool, len(part), amp, True)
        scaler.scale((e.float() * G[i:i + len(part)]).sum()).backward()
    return float(loss), stats

def p7_train(model, teacher, cfg, minutes, amp, log=10):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg.get("wd", 0.0), betas=(0.9, 0.98))
    try: scaler = torch.amp.GradScaler("cuda", enabled=amp and P7_AMP_DTYPE == torch.float16)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=amp and P7_AMP_DTYPE == torch.float16)
    rng = random.Random(P7_SEED + 17 * cfg["arch"]); micro = P7_MICRO; spe = max(1, len(T_VALID) // P7_QBS)
    max_steps = spe * P7_MAX_EPOCHS; budget = minutes * 60; t0 = time.time(); step = 0; ema = acc = None; hist = []; model.eval()
    while step < max_steps and time.time() - t0 < budget:
        prog = max(step / max_steps, (time.time() - t0) / budget)
        lr_now = cfg["lr"] * min(1.0, (step + 1) / 20) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))
        for g in opt.param_groups: g["lr"] = lr_now
        batch = p7_sample(rng, P7_QBS, cfg.get("hn", P7_HN), cfg.get("neg", "hard"))
        try:
            opt.zero_grad(set_to_none=True)
            loss, st = p7_train_step(model, teacher, cfg, batch, micro, amp, scaler)
            if not math.isfinite(loss): opt.zero_grad(set_to_none=True); continue
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
        except torch.cuda.OutOfMemoryError:
            opt.zero_grad(set_to_none=True); p7_free()
            if micro == 1: raise
            micro = max(1, micro // 2); print(f"   OOM -> micro-batch {micro}"); continue
        step += 1; ema = loss if ema is None else 0.95 * ema + 0.05 * loss; acc = st["acc"] if acc is None else 0.95 * acc + 0.05 * st["acc"]
        if step % log == 0 or step == 1:
            hist.append((step, round(ema, 4), round(acc, 3)))
            extra = f" | kd {st['kd']:.3f} emb {st['emb']:.3f}" if "kd" in st else ""
            print(f"   step {step:5d} | epoch {step / spe:4.1f} | loss {ema:.4f} | in-batch top1 {acc:.2f} | lr {lr_now:.2e} | {(time.time()-t0)/60:.1f} min{extra}")
    model.eval()
    return dict(steps=step, epochs=round(step / spe, 2), minutes=round((time.time() - t0) / 60, 1), micro_batch=micro, loss_hist=hist)

In [ ]:
# 11. Runs. One run per Phase-1 architecture, each with a different adaptation technique.
#     Run 1  Arch 4  LoRMA (compulsory)              also the KD teacher
#     Run 2  Arch 3  IA3
#     Run 3  Arch 2  partial / layer-wise fine-tuning (last 4 blocks + final norm)
#     Run 4  Arch 1  knowledge distillation          half-depth student, full fine-tuning, InfoNCE + KL(teacher scores) + embedding alignment
#     Run 5  Arch 4  LoRMA, in-batch negatives only           (ablation: negative sampling)
#     Run 6  Arch 4  LoRMA, pairwise margin loss              (ablation: ranking objective)
#     Run 7  Arch 1  half-depth student, full FT, no teacher  (ablation: what distillation adds)
P7_RUN_CFG = {
    5: dict(name="P7_Run5_Arch4_LoRMA_inbatch",   arch=4, tech="lorma",   r=16,     lr=3e-4, loss="infonce", neg="inbatch", pool="mean"),
    6: dict(name="P7_Run6_Arch4_LoRMA_margin",    arch=4, tech="lorma",   r=16,     lr=3e-4, loss="margin",  neg="hard",    pool="mean")
}
_dev_sample = [ch[0] for ch in DEV_Q_CH[:8]]

def p7_teacher():
    t = p7_make_model(4, "lorma", dict(r=16), "Teacher: Arch 4 + LoRMA (Run 1)")
    f = next(iter(glob.glob(f"{P7_OUT}/P7_Run1_*_trainable.pt") + glob.glob("/kaggle/input/**/P7_Run1_*_trainable.pt", recursive=True)), None)
    assert f, "teacher weights not found: run Run 1 first, or attach the committed output containing P7_Run1_Arch4_LoRMA_trainable.pt"
    res = t.load_state_dict(torch.load(f, map_location="cpu"), strict=False)
    assert not [k for k in res.missing_keys if k.endswith((".A", ".B"))], "teacher adapter weights did not load"
    for p in t.parameters(): p.requires_grad = False
    print("  teacher loaded from", f); return t.eval()

def p7_dev_eval(model, amp):
    Eq = p7_encode(model, DEV_Q_CH, amp); Ed = p7_encode(model, DEV_C_CH, amp)
    res, key, S = p7_eval_combos(Eq, Ed, dev_qids, dev_cids, dev_gold, P7_TOPK)
    sec = (Eq["sec"] + Ed["sec"]) / (len(dev_qids) + len(dev_cids)); idx_mb = Ed["n"] * Ed["mean"].shape[1] * 2 / 1e6
    return dict(table=res, best=key, S=S, sec_per_case=sec, chunks_per_sec=(Eq["n"] + Ed["n"]) / max(1e-9, Eq["sec"] + Ed["sec"]), index_mb=idx_mb, dim=Ed["mean"].shape[1], Eq=Eq, Ed=Ed)

def p7_zeroshot(a):
    f = f"{P7_OUT}/ZS_Arch{a}_metrics.json"
    if P7_SKIP_DONE and os.path.exists(f): return json.load(open(f))
    print(f"\nzero-shot Phase-1 Arch {a} (no adaptation)")
    model = p7_make_model(a, "none", {}, f"ZeroShot_Arch{a}"); tot, _ = p7_count(model)
    amp = p7_amp_ok(model, _dev_sample); ev = p7_dev_eval(model, amp)
    m = dict(arch=a, params_M=round(tot / 1e6, 2), best=ev["best"], dev=ev["table"][ev["best"]], table=ev["table"], sec_per_case=round(ev["sec_per_case"], 4), dry=P7_DRY_RUN)
    print("  best combo", m["best"], m["dev"]); json.dump(m, open(f, "w"), indent=1); del model; p7_free(); return m

def p7_run(rid):
    cfg = P7_RUN_CFG[rid]; name = cfg["name"]; sub = f"{P7_OUT}/submission_{name}.zip"; mf = f"{P7_OUT}/{name}_metrics.json"
    if P7_SKIP_DONE and os.path.exists(sub) and os.path.exists(mf):
        print(f"[{name}] already done, skipping"); return json.load(open(mf))
    print("\n" + "=" * 100 + f"\n{name}: Arch {cfg['arch']} + {cfg['tech']} | loss {cfg['loss']} | negatives {cfg['neg']}\n" + "=" * 100)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    model = p7_make_model(cfg["arch"], cfg["tech"], cfg, name); tot, trn = p7_count(model)
    print(f"  parameters: {tot/1e6:.1f}M total | {trn/1e6:.3f}M trainable ({100*trn/tot:.1f}%)")
    amp = p7_amp_ok(model, _dev_sample)
    teacher = p7_teacher() if cfg["tech"] == "kd" else None
    info = p7_train(model, teacher, cfg, 3 if P7_DRY_RUN else P7_MINUTES[rid], amp)
    del teacher; p7_free()
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    ev = p7_dev_eval(model, amp); key = ev["best"]; dev_best = ev["table"][key]
    print(f"  DEV best combination {key}: {dev_best}")
    lam, f1h, prh, lam_tab = p7_hybrid_tune(ev["S"], S_bm_dev, dev_qids, dev_cids, dev_gold, P7_TOPK)
    print(f"  DEV hybrid (dense + {lam} x BM25): F1 {f1h:.4f} | BM25 alone {BM25_DEV['F1']:.4f}")
    Tq = p7_encode(model, TEST_Q_CH, amp); Td = p7_encode(model, TEST_C_CH, amp)
    S_te = p7_combo_scores(Tq, Td, key); cset = set(test_cids)
    p7_write_submission(sub, p7_topk(S_te, test_qids, test_cids, P7_TOPK), raw_test_q, raw_test_c, test_qids, cset)
    subh = f"{P7_OUT}/submission_{name}_hybrid.zip"
    p7_write_submission(subh, p7_topk(p7_z(S_te) + lam * p7_z(S_bm_test), test_qids, test_cids, P7_TOPK), raw_test_q, raw_test_c, test_qids, cset)
    np.savez_compressed(f"{P7_OUT}/{name}_scores.npz", dev=ev["S"].astype(np.float32), test=S_te.astype(np.float32))
    if cfg["tech"] not in ("kd", "full"):
        torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, f"{P7_OUT}/{name}_trainable.pt")
    m = dict(run=name, arch=cfg["arch"], technique=cfg["tech"], loss=cfg["loss"], negatives=cfg["neg"], params_total_M=round(tot / 1e6, 2), params_trainable_M=round(trn / 1e6, 3),
             peak_gpu_gb=round(peak, 2), amp=amp, train=info, best_combo=key, dev=dev_best, dev_table=ev["table"], hybrid=dict(lam=lam, F1=round(f1h, 4), lam_table=lam_tab),
             bm25_dev=BM25_DEV, dim=ev["dim"], sec_per_case=round(ev["sec_per_case"], 4), chunks_per_sec=round(ev["chunks_per_sec"], 1), index_mb_dev_pool=round(ev["index_mb"], 2),
             n_chunks=P7_MAX_CHUNKS, ctx=CTX, dry=P7_DRY_RUN)
    json.dump(m, open(mf, "w"), indent=1, default=str); print(f"  wrote {sub} and {subh}")
    del model, ev, Tq, Td; p7_free(); return m

P7_RESULTS = {}
for _rid in sorted(P7_RUNS): P7_RESULTS[_rid] = p7_run(_rid)
P7_ZS = {}
if P7_ZEROSHOT:
    for _a in sorted({P7_RUN_CFG[r]["arch"] for r in P7_RUNS}, reverse=True):
        if globals().get(f"ARCH{_a}_CHECKPOINT"): P7_ZS[_a] = p7_zeroshot(_a)

In [ ]:
# 12. Comparison table, analysis notes, failure analysis, score-level ensemble, submission check
#     when their committed output is attached (Add Input).
import pandas as pd
pd.set_option("display.width", 260, "display.max_columns", 40)
def _find_all(pat): return glob.glob(f"{P7_OUT}/{pat}") + glob.glob(f"/kaggle/input/**/p7_out/{pat}", recursive=True)
def _find_one(fname): return next(iter(_find_all(fname)), None)

_ms, _zs = {}, {}
for f in _find_all("P7_Run*_metrics.json"):
    m = json.load(open(f))
    if bool(m.get("dry")) == P7_DRY_RUN: _ms.setdefault(m["run"], m)
for f in _find_all("ZS_Arch*_metrics.json"):
    m = json.load(open(f))
    if bool(m.get("dry")) == P7_DRY_RUN: _zs.setdefault(m["arch"], m)
assert _ms, "no finished run found in the output folder"

rows = []
for n, m in sorted(_ms.items()):
    d = m["dev"]; z = _zs.get(m["arch"], {}).get("dev", {}).get("F1")
    rows.append({"run": n, "arch": m["arch"], "technique": m["technique"], "loss": m["loss"], "negs": m["negatives"], "params_M": m["params_total_M"], "trainable_M": m["params_trainable_M"],
                 "peak_GB": m["peak_gpu_gb"], "train_min": m["train"]["minutes"], "steps": m["train"]["steps"], "best_combo": m["best_combo"],
                 "P@7": d["P"], "R@7": d["R"], "F1@7": d["F1"], "hybrid_F1": m["hybrid"]["F1"], "zeroshot_F1": z, "gain": None if z is None else round(d["F1"] - z, 4),
                 "dim": m["dim"], "ms/case": round(1000 * m["sec_per_case"], 1), "chunks/s": m["chunks_per_sec"], "index_MB": m["index_mb_dev_pool"]})
df = pd.DataFrame(rows); print(df.to_string(index=False))
print("\nBM25 reference on Dev (no neural weights):", {k: round(v, 4) for k, v in BM25_DEV.items()}, "| best (k1, b) =", BM25_PARAMS)
if _zs: print("Zero-shot Phase-1 models on Dev (F1@7, best combo):", {f"Arch{a}": (z["dev"]["F1"], z["best"]) for a, z in sorted(_zs.items())})

try: _tbl = df.to_markdown(index=False)
except Exception: _tbl = df.to_string(index=False)
lines = ["# Phase 7 analysis notes (numbers from the Dev split; Micro-F1@7 is the primary metric)\n", _tbl, ""]
lines.append(f"- BM25 baseline on Dev: P/R/F1@7 = {BM25_DEV['P']:.4f} / {BM25_DEV['R']:.4f} / {BM25_DEV['F1']:.4f} (k1, b = {BM25_PARAMS}).")
best = max(_ms.values(), key=lambda m: m["dev"]["F1"])
lines.append(f"- Best dense run on Dev: {best['run']} with {best['best_combo']} -> F1@7 {best['dev']['F1']}; with BM25 fusion (lambda {best['hybrid']['lam']}) {best['hybrid']['F1']}.")

# --- effect of pooling / centring / aggregation (marginal means over all runs, and over the zero-shot models) ---
def _marg(tables, idx):
    agg = collections.defaultdict(list)
    for t in tables:
        for k, v in t.items(): agg[k.split("|")[idx]].append(v["F1"])
    return {k: round(float(np.mean(v)), 4) for k, v in sorted(agg.items())}
for title, tabs in (("adapted runs", [m["dev_table"] for m in _ms.values()]), ("zero-shot Phase-1", [z["table"] for z in _zs.values()])):
    if not tabs: continue
    for idx, nm in enumerate(("pooling", "centring", "chunk aggregation")):
        lines.append(f"- Mean Dev F1@7 by {nm} ({title}): {_marg(tabs, idx)}")

# --- technique and architecture comparison, gain over the untouched Phase-1 model ---
for r in rows:
    if r["gain"] is not None: lines.append(f"- {r['run']}: F1@7 {r['F1@7']} vs zero-shot Arch{r['arch']} {r['zeroshot_F1']} (gain {r['gain']:+.4f}); trainable {r['trainable_M']}M of {r['params_M']}M parameters; {r['train_min']} min / {r['steps']} steps; peak {r['peak_GB']} GB.")

# --- knowledge distillation ---
r1, r4, r7 = (_ms.get(n) for n in ("P7_Run1_Arch4_LoRMA", "P7_Run4_Arch1_KD", "P7_Run7_Arch1_FullFT_noKD"))
if r1 and r4:
    lines.append(f"- KD: student {r4['params_total_M']}M params vs teacher {r1['params_total_M']}M ({r4['params_total_M'] / r1['params_total_M']:.0%}); {1000 * r4['sec_per_case']:.1f} vs {1000 * r1['sec_per_case']:.1f} ms/case; Dev F1@7 {r4['dev']['F1']} vs {r1['dev']['F1']} "
                 f"({r4['dev']['F1'] / max(1e-9, r1['dev']['F1']):.0%} of the teacher); peak training memory {r4['peak_gpu_gb']} vs {r1['peak_gpu_gb']} GB.")
if r4 and r7: lines.append(f"- Distillation vs the same student without a teacher: F1@7 {r4['dev']['F1']} vs {r7['dev']['F1']}.")
for a, b, what in (("P7_Run1_Arch4_LoRMA", "P7_Run5_Arch4_LoRMA_inbatch", "hard negatives vs in-batch negatives only"), ("P7_Run1_Arch4_LoRMA", "P7_Run6_Arch4_LoRMA_margin", "InfoNCE vs pairwise margin")):
    if a in _ms and b in _ms: lines.append(f"- {what}: F1@7 {_ms[a]['dev']['F1']} vs {_ms[b]['dev']['F1']}.")

# --- failure analysis on the best dense run ---
_f = _find_one(f"{best['run']}_scores.npz")
if _f:
    sc = np.load(_f)["dev"]; pred = p7_topk(sc, dev_qids, dev_cids, P7_TOPK); cpos = {c: i for i, c in enumerate(dev_cids)}
    qlen = np.asarray([len(w) for w in dev_qw]); ng = np.asarray([len(dev_gold[q]) for q in dev_qids]); f1q, ranks, miss = [], [], 0
    for qi, q in enumerate(dev_qids):
        g = set(dev_gold[q]); tp = len(set(pred[q]) & g); f1q.append(2 * tp / (P7_TOPK + max(1, len(g))))
        for c in g:
            j = cpos.get(c)
            if j is None: miss += 1
            else: ranks.append(int((sc[qi] > sc[qi, j]).sum()) + 1)
    f1q = np.asarray(f1q); ranks = np.asarray(ranks)
    def _bucket(x, name):
        qs = np.percentile(x, [25, 50, 75]); b = np.digitize(x, qs)
        return f"mean per-query F1@7 by {name} quartile (shortest -> longest / fewest -> most): " + ", ".join(f"{f1q[b == k].mean():.3f}" for k in range(4) if (b == k).any())
    lines.append(f"- Failure analysis ({best['run']}): " + _bucket(qlen, "query length") + "; " + _bucket(ng, "number of gold cases") + ".")
    if len(ranks): lines.append(f"- Gold cases: median rank {int(np.median(ranks))}, in top 7 {np.mean(ranks <= 7):.1%}, in top 20 {np.mean(ranks <= 20):.1%}, in top 50 {np.mean(ranks <= 50):.1%}; {np.mean(f1q == 0):.1%} of queries retrieve no gold case in the top 7; {miss} gold ids are not in the Dev pool.")

# --- score-level ensemble of the finished runs (+ BM25), tuned on Dev ---
_sc = {}
for n in _ms:
    f = _find_one(f"{n}_scores.npz")
    if f: z = np.load(f); _sc[n] = (z["dev"], z["test"])
if len(_sc) >= 2:
    zd = np.mean([p7_z(v[0]) for v in _sc.values()], 0); zt = np.mean([p7_z(v[1]) for v in _sc.values()], 0)
    lam, f1e, _, tab = p7_hybrid_tune(zd, S_bm_dev, dev_qids, dev_cids, dev_gold, P7_TOPK)
    zero = tab.get(0.0)
    print(f"\nEnsemble of {list(_sc)}: Dev F1@7 dense-only {zero} | with BM25 (lambda {lam}) {f1e:.4f} | lambda table {tab}")
    p7_write_submission(f"{P7_OUT}/submission_P7_Ens_all.zip", p7_topk(p7_z(zt) + lam * p7_z(S_bm_test), test_qids, test_cids, P7_TOPK), raw_test_q, raw_test_c, test_qids, set(test_cids))
    lines.append(f"- Score-level ensemble of {len(_sc)} runs: Dev F1@7 {zero} dense-only, {f1e:.4f} with BM25 (lambda {lam}); written to submission_P7_Ens_all.zip (extra, beyond the four required runs).")

lines += ["", "## Notes to expand in the report (items of section 13.5)",
          "- Phase-1 representation transfer: compare each run with its own zero-shot row; the Phase-1 objective was next-token prediction on legal text, so last-token / mean states are not trained to be comparable across documents.",
          "- Long cases: only P7_MAX_CHUNKS x 512 selected tokens are encoded; see the aggregation marginals and the query-length buckets above.",
          "- BM25 hard negatives are lexically close but not guaranteed to be truly non-relevant (labels are citations only); the skipped top ranks and the exclusion of known positives limit that risk.",
          "- Different Phase-1 architectures are paired with different techniques, so technique and backbone are confounded; the zero-shot rows give the per-backbone starting point."]
open(f"{P7_OUT}/analysis_summary.md", "w").write("\n".join(lines)); print("\n" + "\n".join(lines[2:]))

# --- submission check ---
_cset, _qset = set(test_cids), set(test_qids)
for z in sorted(glob.glob(f"{P7_OUT}/submission_*.zip")):
    p = json.load(zipfile.ZipFile(z).open("prediction.json")); ok = {str(k) for k in p} == _qset and all(len(v) == P7_TOPK and len(set(v)) == P7_TOPK and all(str(i) in _cset for i in v) for v in p.values())
    print(f"{os.path.basename(z)}: {len(p)} queries | 7 unique ids from the test pool for every query: {ok}")
print("\nSubmit the four dense zips submission_P7_Run1..Run4 (or their _hybrid versions); the ensemble and ablation zips are extras.")